In [ ]:
import cv2
import face_recognition
import os
from key import Button,language
from PIL import Image, ImageDraw
import xgoscreen.LCD_2inch as LCD_2inch
splash_theme_color = (255,255,255)
mydisplay = LCD_2inch.LCD_2inch()
mydisplay.Init()
mydisplay.clear()
# Init Splash
splash = Image.new("RGB", (mydisplay.height, mydisplay.width), splash_theme_color)
draw = ImageDraw.Draw(splash)
mydisplay.ShowImage(splash)
button = Button()

la = language()

In [ ]:
folder_path = './Face_P/'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

from picamera2 import Picamera2
picam2 = Picamera2()
picam2.configure(picam2.create_preview_configuration(main={"format": 'RGB888', "size": (320, 240)}))
picam2.start()

In [ ]:
import ipywidgets.widgets as widgets
from IPython.display import display
#显示摄像头组件 Display camera components
image_widget = widgets.Image(format='jpeg', width=320, height=240)

#bgr8转jpeg格式  bgr8 to jpeg format
import enum
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])


In [ ]:
## 先拍照，再识别 Take a photo first, then identify
import time  
def take_photo():
    global i 
    show_ok = False  # 控制是否显示OK字样 Control whether the word 'OK' is displayed
    ok_display_time = 0  # 记录显示OK的时间 Record the time when OK is displayed
    
    while True:
        frame = picam2.capture_array()
        
        # 如果需要显示OK，在画面上添加文字 If OK needs to be displayed, add text on the screen
        if show_ok:
            current_time = time.time()
            if current_time - ok_display_time < 1.0: 
                font = cv2.FONT_HERSHEY_SIMPLEX
                text = "OK" 
                text_size = cv2.getTextSize(text, font, 2, 3)[0]
                text_x = (frame.shape[1] - text_size[0]) // 2
                cv2.putText(frame, text, (text_x, 50), font, 2, (0, 255, 0), 3, cv2.LINE_AA)
            else:
                show_ok = False  # 1秒后停止显示 Stop displaying after 1 second
        
        #cv2.imshow('frame', frame)
        image_widget.value = bgr8_to_jpeg(frame)
        
        b, g, r = cv2.split(frame)
        img = cv2.merge((r, g, b))
        imgok = Image.fromarray(img)
        mydisplay.ShowImage(imgok) 
        
        if button.press_d():
            cv2.imwrite(f"./Face_P/{i}.jpg", frame)
            i += 1
            # 设置显示OK标志和时间
            show_ok = True
            ok_display_time = time.time()
            
            if la == "cn":
                print("拍照成功")
            else:
                print("Photo taken successfully")
                
        if button.press_c():
            if la=="cn":
                print("退出人脸录入模式，开始识别")
            else:
                print("Exit face input mode and start recognition")
            break
            
        if button.press_b() :
            exit()

In [ ]:
def load_photo(i):
    total_image_name = []
    total_face_encoding = []
    for j in range(1, i): 
        fn = f"./Face_P/{j}.jpg"  # 这里是拍照后保存的图片名 Here is the name of the picture saved after taking a photo
        print(fn)
        # try:
        total_face_encoding.append(face_recognition.face_encodings(face_recognition.load_image_file(fn))[0])
        # except:
        #     print(f"人脸识别失败，图片{fn}中人脸识别不清晰或不存在")
        #     print(f"Facial recognition failed, the facial recognition in image {fn} is unclear or non-existent")
        fn = fn[:(len(fn) - 4)]  #截取图片名 Extract the image name 
        total_image_name.append(fn)  #图片名字列表 List of Image Names
    return total_image_name, total_face_encoding

In [ ]:
display(image_widget)
face = False
while (1):
    while True:
        if face == True:
            break
        if la == "cn":
            print("开始人脸录入")
        else:
            print("Start facial recognition")
        i = 1
        take_photo()
        try:
            total_image_name, total_face_encoding = load_photo(i)
            face = True
            print("Face input successful")
            break
        except Exception as e:
            print("Face input Fail")
            continue
    frame = picam2.capture_array()
   
    face_locations = face_recognition.face_locations(frame)
    face_encodings = face_recognition.face_encodings(frame, face_locations)
    # 在这个视频帧中循环遍历每个人脸 Loop through each face in this video frame
    for (top, right, bottom, left), face_encoding in zip(
            face_locations, face_encodings):
        # 看看面部是否与已知人脸相匹配。 Check if the face matches a known face.
        for i, v in enumerate(total_face_encoding):
            match = face_recognition.compare_faces(
                [v], face_encoding, tolerance=0.5)
            name = "Unknown"
            if match[0]:
                total_image_name[i] = total_image_name[i].replace(folder_path,"")
                name = total_image_name[i] #str(i+1) 
                name = 'NO.' + name
                break
        # 画出一个框，框住脸 Draw a frame to enclose the face
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)
        # 画出一个带名字的标签，放在框下 Draw a label with a name and place it under the box
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), (0, 0, 255),
                      cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        try:
            cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.7,
                    (255, 255, 255), 1)
        except :
            name = "Unknown"
            cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.7,
                    (255, 255, 255), 1)
    # 显示结果图像 Display result image
    b, g, r = cv2.split(frame)
    img = cv2.merge((r, g, b))
    imgok = Image.fromarray(img)
    mydisplay.ShowImage(imgok)  
    image_widget.value = bgr8_to_jpeg(frame)
    if button.press_b():
        break

